In [ ]:
# -*- coding: utf-8 -*-
from Library import utils, dataset
import os
import gc
import json
import h5py
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from scipy.signal import detrend
from scipy.stats import gaussian_kde

# ==============================================================================
# KONFIGURASI & SETUP
# ==============================================================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# --- PATH & PARAMETER ---
CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/STEAD_benchmarking_data/mcquake_ori_file/Code & Figure demo"
MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

BUFFER_SIZE = 32
N_SAMPLES = 10000
DECISION_BIAS = 1.0  # Faktor untuk menekan False Positive (makin tinggi makin konservatif)
RANDOM_SEED = 42

# --- Buat folder penyimpanan hasil ---
SAVE_BASE = '/Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/stead_hdf5_10k_metal_corrected'
os.makedirs(SAVE_BASE, exist_ok=True)
time_str = datetime.now().strftime("%d%H%M%S")
save_dir = os.path.join(SAVE_BASE, f"MCU_Quake_1C_STEAD_HDF5_{time_str}")
os.makedirs(save_dir, exist_ok=True)

# Tambahkan file handler untuk log
file_handler = logging.FileHandler(os.path.join(save_dir, "task_log.txt"))
file_handler.setFormatter(logging.Formatter('%(asctime)s - [%(levelname)s]: %(message)s'))
logger.addHandler(file_handler)

# ==============================================================================
# FUNGSI PENDUKUNG
# ==============================================================================
def preprocess_waveform(waveform, p_arrival=None, is_noise=False, num_points=700, norm_points=900):
    wf = detrend(waveform, type='linear')
    if is_noise:
        signal = wf[:num_points]
        norm_val = np.max(np.abs(wf[:norm_points]))
        return signal / (norm_val if norm_val > 1e-8 else 1.0)
    else:
        if p_arrival is None:
            return None
        p = int(p_arrival)
        if p + num_points > len(wf) or p + norm_points > len(wf):
            return None
        signal = wf[p:p + num_points]
        norm_val = np.max(np.abs(wf[p:p + norm_points]))
        return signal / (norm_val if norm_val > 1e-8 else 1.0)

def predict_1c_batch(model, batch_waves, kde_noise, kde_le, bias):
    batch_input = tf.convert_to_tensor(np.array(batch_waves, dtype=np.float32).reshape(-1, 700, 1))
    output = model(batch_input)
    # TFSMLayer bisa mengembalikan dict atau Tensor
    if isinstance(output, dict):
        # Ambil nilai pertama (biasanya satu-satunya output)
        embeddings = list(output.values())[0].numpy().flatten()
    else:
        embeddings = output.numpy().flatten()
    
    prob_noise = kde_noise.pdf(embeddings)
    prob_le = kde_le.pdf(embeddings)
    # Gunakan bias untuk menekan LE (kurangi FP)
    return (prob_le > (prob_noise * bias)).astype(int)

# ==============================================================================
# MAIN PROSES
# ==============================================================================
if __name__ == "__main__":
    logger.info("Memuat model (TFSMLayer)...")
    embedding_layer = tf.keras.layers.TFSMLayer(MODEL_PATH, call_endpoint='serving_default')
    
    # Load Embedding (KDE) – menggunakan komponen Z saja sesuai skrip asli
    emb_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    keys = list(emb_Z.keys())
    k_noise = next((k for k in keys if k.lower() in ['noise','no']), keys[0])
    k_le = next((k for k in keys if k.lower() in ['le','earthquake','eq']), keys[1])
    
    kde_noise = gaussian_kde(np.array(emb_Z[k_noise]).T)
    kde_le = gaussian_kde(np.array(emb_Z[k_le]).T)
    
    # Auto-Fix label KDE jika tertukar (mean noise > mean LE)
    if kde_noise.dataset.mean() > kde_le.dataset.mean():
        logger.warning("KDE label tertukar. Menukar label KDE...")
        kde_noise, kde_le = kde_le, kde_noise
    
    # Data Preparation
    logger.info("Membaca CSV...")
    df = pd.read_csv(CSV_PATH, low_memory=False)
    df = df[df['trace_category'].isin(['earthquake_local', 'noise'])]
    
    # Sampling seimbang dengan seed
    np.random.seed(RANDOM_SEED)
    n_per_class = N_SAMPLES // 2
    df_eq = df[df['trace_category'] == 'earthquake_local']
    df_no = df[df['trace_category'] == 'noise']
    
    # Jika jumlah sampel kurang dari n_per_class, ambil semua
    n_eq = min(n_per_class, len(df_eq))
    n_no = min(n_per_class, len(df_no))
    df_eq_sample = df_eq.sample(n=n_eq, random_state=RANDOM_SEED)
    df_no_sample = df_no.sample(n=n_no, random_state=RANDOM_SEED)
    df_final = pd.concat([df_eq_sample, df_no_sample]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    logger.info(f"Total sampel uji: {len(df_final)} (EQ: {n_eq}, NO: {n_no})")

    # Inferensi
    all_true, all_pred = [], []
    buffer_waves = []
    
    with h5py.File(HDF5_PATH, 'r') as h5:
        data_group = h5['data']
        for idx in tqdm(range(len(df_final)), desc="Inferensi"):
            row = df_final.iloc[idx]
            trace = row['trace_name']
            cat = row['trace_category']
            if trace not in data_group:
                continue
            
            raw = data_group[trace][:, 2]  # Z component
            if cat == 'earthquake_local':
                p = row['p_arrival_sample']
                if pd.isna(p):
                    continue
                signal = preprocess_waveform(raw, p_arrival=p, is_noise=False)
                true_label = 1
            else:  # noise
                signal = preprocess_waveform(raw, is_noise=True)
                true_label = 0
            
            if signal is None:
                continue
            
            buffer_waves.append(signal)
            all_true.append(true_label)
            
            if len(buffer_waves) >= BUFFER_SIZE:
                preds = predict_1c_batch(embedding_layer, buffer_waves, kde_noise, kde_le, DECISION_BIAS)
                all_pred.extend(preds)
                buffer_waves = []  # kosongkan buffer
                gc.collect()  # bersihkan memori jika perlu
        
        # Proses sisa buffer
        if buffer_waves:
            preds = predict_1c_batch(embedding_layer, buffer_waves, kde_noise, kde_le, DECISION_BIAS)
            all_pred.extend(preds)
            buffer_waves = []

    # --- METRIK ---
    y_true = np.array(all_true, dtype=np.int32)
    y_pred = np.array(all_pred, dtype=np.int32)
    TP = np.sum((y_true==1)&(y_pred==1))
    TN = np.sum((y_true==0)&(y_pred==0))
    FP = np.sum((y_true==0)&(y_pred==1))
    FN = np.sum((y_true==1)&(y_pred==0))
    total = TP+TN+FP+FN
    acc = (TP+TN)/total if total>0 else 0
    tpr_le = TP/(TP+FN) if (TP+FN)>0 else 0
    tnr_no = TN/(TN+FP) if (TN+FP)>0 else 0
    ppv_le = TP/(TP+FP) if (TP+FP)>0 else 0
    ppv_no = TN/(TN+FN) if (TN+FN)>0 else 0
    f1 = 2*ppv_le*tpr_le/(ppv_le+tpr_le) if (ppv_le+tpr_le)>0 else 0

    metrics_1C = {
        'accuracy (avg.)': acc,
        'True positive rate (avg.)': (tpr_le + tnr_no) / 2,
        'Positive predictive value (avg.)': (ppv_le + ppv_no) / 2,
        'False positive rate (avg.)': 1 - tnr_no,
        'F1-score (avg.)': f1
    }

    # --- VISUALISASI (menggunakan save_dir yang sudah didefinisikan) ---
    cm = np.array([[TN, FP], [FN, TP]])
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.matshow(cm, cmap=plt.cm.Blues, vmin=0, vmax=np.max(cm))
    threshold = np.max(cm)/2.
    for i in range(2):
        for j in range(2):
            color = "white" if cm[i,j] > threshold else "#08306b"
            ax.text(j, i, format(cm[i,j], 'd'), ha='center', va='center', color=color, fontsize=15, weight='bold')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['NO.', 'LE.'], fontsize=13, weight='bold')
    ax.set_yticklabels(['NO.', 'LE.'], fontsize=13, weight='bold')
    ax.xaxis.set_ticks_position('top')
    ax.set_xlabel('Predicted', fontsize=13, weight='bold', labelpad=10)
    ax.set_ylabel('True', fontsize=13, weight='bold', labelpad=10)
    ax.set_title(f'MCU_Quake_1C STEAD_HDF5 (n={total})', fontsize=14, weight='bold')
    ax.text(1.7, -0.4, 'TPR:', ha='center', va='center', fontsize=12, weight='bold', clip_on=False)
    ax.text(1.7, 0, f'{tnr_no*100:.2f}%', ha='center', va='center', fontsize=12, weight='bold', color='#c92a2a', clip_on=False)
    ax.text(1.7, 1, f'{tpr_le*100:.2f}%', ha='center', va='center', fontsize=12, weight='bold', color='blue', clip_on=False)
    ax.text(-0.4, 1.7, 'PPV:', ha='center', va='center', fontsize=12, weight='bold', clip_on=False)
    ax.text(0, 1.7, f'{ppv_no*100:.2f}%', ha='center', va='center', fontsize=12, weight='bold', color='black', clip_on=False)
    ax.text(1, 1.7, f'{ppv_le*100:.2f}%', ha='center', va='center', fontsize=12, weight='bold', color='black', clip_on=False)
    plt.tight_layout()
    cm_path = os.path.join(save_dir, "STEAD_HDF5_CM.png")
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.close()
    logger.info(f"Confusion matrix tersimpan: {cm_path}")

    def plot_comprehensive_metrics(metrics, save_path):
        labels = ['Accuracy', 'Recall', 'Precision', 'FPR', 'F1-Score']
        keys = ['accuracy (avg.)', 'True positive rate (avg.)', 'Positive predictive value (avg.)', 'False positive rate (avg.)', 'F1-score (avg.)']
        values = [metrics.get(k, 0) for k in keys]
        plt.figure(figsize=(12, 6))
        bars = plt.bar(labels, values, color=['#2c3e50', '#3498db', '#e67e22', '#e74c3c', '#27ae60'])
        plt.ylim(0, 1.0)
        plt.ylabel('Score')
        plt.title('Performance Metrics Evaluation (Average)')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', va='bottom')
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close()
        logger.info(f"Bar chart tersimpan: {save_path}")

    bar_path = os.path.join(save_dir, "STEAD_HDF5_metrics_bar.png")
    plot_comprehensive_metrics(metrics_1C, bar_path)

    with open(os.path.join(save_dir, "metrics.json"), 'w') as f:
        json.dump(metrics_1C, f, indent=2)

    logger.info("\n" + "="*40)
    logger.info(f"STEAD HDF5 10K ACCURACY: {acc:.4f}")
    logger.info(f"STEAD HDF5 10K F1-SCORE: {f1:.4f}")
    logger.info("="*40)
    logger.info(f"Hasil disimpan di: {save_dir}")
    logger.info("Pengujian STEAD HDF5 SELESAI (menggunakan Metal).")

Memuat model (TFSMLayer)...
Membaca CSV...
Total sampel uji: 10000 (EQ: 5000, NO: 5000)
Inferensi: 100%|██████████| 10000/10000 [05:20<00:00, 31.24it/s]
Confusion matrix tersimpan: /Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/stead_hdf5_10k_metal_corrected/MCU_Quake_1C_STEAD_HDF5_19144653/STEAD_HDF5_CM.png
Bar chart tersimpan: /Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/stead_hdf5_10k_metal_corrected/MCU_Quake_1C_STEAD_HDF5_19144653/STEAD_HDF5_metrics_bar.png

STEAD HDF5 10K ACCURACY: 0.4922
STEAD HDF5 10K F1-SCORE: 0.6575
Hasil disimpan di: /Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/stead_hdf5_10k_metal_corrected/MCU_Quake_1C_STEAD_HDF5_19144653
Pengujian STEAD HDF5 SELESAI (menggunakan Metal).
